# 1kg_eur — Rare Variant GRM (MAF 0.01 – 1 %, Batch)

Builds a genomic relatedness matrix restricted to rare variants: MAF > 0.01 % and < 1 %.
These variants are **not** in the common-variant BED produced by `04_grm_panel_qc.ipynb`
(which applies `--maf 0.01` as a lower bound).

QC is more stringent than for common variants: tighter genotype call rate, mild HWE guard,
and biallelic-only filter.

**Step 1** — single Batch task downloads the unified panel PGEN from GCS, applies plink2 QC,
produces a BED, computes allele frequencies with plink 1.9, and uploads the result.

**Step 2** — sharded Batch jobs compute the GRM in parallel (same pattern as `06_grm_shards.ipynb`).

## Config

In [ ]:
import math, os, subprocess

PROJECT_ID      = "wb-swift-sprout-7231"
REGION          = "us-central1"
SERVICE_ACCOUNT = "pet-27799165194323faf22e2@wb-swift-sprout-7231.iam.gserviceaccount.com"
NETWORK         = f"projects/{PROJECT_ID}/global/networks/network"
SUBNETWORK      = f"projects/{PROJECT_ID}/regions/{REGION}/subnetworks/subnetwork"
CLOUD_SDK_TAG   = "581.0.0-slim"

WS_GS  = "gs://cloned-shared-env-pilot-wb-swift-sprout-7231/phenotypic_covariance_v9"
R_GS   = f"{WS_GS}/1kg_eur"

# ── raw unified panel (shared across sample sets) ───────────────────────────────
PANEL_GS = f"{WS_GS}/01_ancestry_filtering/unified_panel/unified_panel_v9"
# GCS keep list (written under old name pre-rename; same sample set)
KEEP_GS  = f"{R_GS}/01_ancestry/round2/1kg_eur_keep_ids.txt"

# ── plink binaries staged in GCS ────────────────────────────────────────────────
PLINK2_BIN_GS = f"{R_GS}/03_grm/bin/plink2"   # stage with cell below if not present
PLINK_BIN_GS  = f"{R_GS}/03_grm/bin/plink"    # plink 1.9 staged in 06_grm_shards

# ── output ──────────────────────────────────────────────────────────────────────
BED_RARE_NAME = "1kg_CEUGBR_GRM_rare"
RARE_OUT_GS   = f"{R_GS}/03_grm/grm_input_rare"
SHARD_RARE_GS = f"{R_GS}/03_grm/shards_rare"
LOG_GS        = f"{R_GS}/03_grm/logs_rare"

# ── filter job sizing ───────────────────────────────────────────────────────────
# Disk must hold: ~116 GB pgen + ~313 MB pvar + output BED (unknown) + working space
FILTER_MACHINE = "n1-standard-8"   # 30 GB RAM; plink2 QC is I/O-bound
FILTER_DISK_GB = 500               # 116 GB in + estimated ~100 GB out + headroom

# ── GRM shard sizing ────────────────────────────────────────────────────────────
# Set BED_SIZE_RARE_GB from the preflight check output after step 1 completes:
#   gcloud storage ls -l {RARE_OUT_GS}/{BED_RARE_NAME}.bed | awk '{{printf "%.1f\n", $1/1024/1024/1024}}'
# WARNING: rare-variant BED can be large (potentially 100-300 GB depending on variant count).
# Memory per shard = (BED_SIZE_RARE_GB * 2 + 4) * 1024 MB.
BED_SIZE_RARE_GB = 100   # placeholder — update after step 1
SHARD_VCPUS      = 16
_raw_mb   = int((BED_SIZE_RARE_GB * 2 + 4) * 1024)
MEMORY_MB = math.ceil(_raw_mb / 256) * 256
_mem_per_vcpu = MEMORY_MB // SHARD_VCPUS
SHARD_MACHINE = (f"n1-custom-{SHARD_VCPUS}-{MEMORY_MB}-ext"
                 if _mem_per_vcpu > 8192
                 else f"n1-custom-{SHARD_VCPUS}-{MEMORY_MB}")
PLINK_MEM_MB  = MEMORY_MB - 8192
SHARD_DISK_GB = 200

N_SHARDS = 20
N_TASKS  = 5

for k, v in dict(
    BED_RARE_NAME=BED_RARE_NAME,
    RARE_OUT_GS=RARE_OUT_GS,
    SHARD_RARE_GS=SHARD_RARE_GS,
    SHARD_MACHINE=SHARD_MACHINE,
    MEMORY_MB=MEMORY_MB,
    PLINK_MEM_MB=PLINK_MEM_MB,
    N_SHARDS=N_SHARDS,
    N_TASKS=N_TASKS,
).items():
    print(f"  {k}: {v}")

## Stage plink2 binary

Run once. Copies the locally installed plink2 (from `04_grm_panel_qc.ipynb` cell 4)
to GCS so the Batch filter job can use it as an `--input`.
Skip if already staged.

In [ ]:
subprocess.run(["bash", "-c", f"""
if gcloud storage ls "{PLINK2_BIN_GS}" >/dev/null 2>&1; then
  echo "already staged: {PLINK2_BIN_GS}"
else
  PLINK2="$HOME/bin/plink2"
  [ -x "$PLINK2" ] || {{ echo "plink2 not installed — run 04_grm_panel_qc cell 4 first"; exit 1; }}
  gcloud storage cp "$PLINK2" "{PLINK2_BIN_GS}"
  echo "staged: {PLINK2_BIN_GS}"
fi
"""], check=True)

## Install dsub

In [ ]:
subprocess.run(["bash", "-c", f"""
pip install --quiet --upgrade 'dsub>=0.5.3'
DSUB_DIR=$(python -c "import dsub, os; print(os.path.dirname(dsub.__file__))")
sed -i -E \
  "s|cloud-sdk:[0-9]+\\.[0-9]+\\.[0-9]+-slim|cloud-sdk:{CLOUD_SDK_TAG}|g" \
  "$DSUB_DIR/providers/google_utils.py"
echo "dsub: $(dsub --version)"
echo "wrapper image: $(grep CLOUD_SDK_IMAGE $DSUB_DIR/providers/google_utils.py)"
"""], check=True)

## Step 1: QC and filter to rare variants (MAF 0.01 – 1 %)

Single Batch task:
1. Downloads the unified panel PGEN (~116 GB) and the round-2 keep list.
2. Applies plink2 QC: MAF 0.01 – 1 %, genotype call rate > 99 %, mild HWE guard, biallelic only.
3. Recomputes allele frequencies with plink 1.9 `.frq` format (for GRM shard jobs).
4. Uploads BED + freq to `grm_input_rare/`.

In [ ]:
subprocess.run(["bash", "-c", f"""
set -eo pipefail
DSUB_DIR=$(python -c "import dsub, os; print(os.path.dirname(dsub.__file__))")
sed -i -E \
  "s|cloud-sdk:[0-9]+\\.[0-9]+\\.[0-9]+-slim|cloud-sdk:{CLOUD_SDK_TAG}|g" \
  "$DSUB_DIR/providers/google_utils.py"

dsub \
  --provider google-batch --project "{PROJECT_ID}" --regions "{REGION}" \
  --logging "{LOG_GS}" \
  --service-account "{SERVICE_ACCOUNT}" \
  --network "{NETWORK}" --subnetwork "{SUBNETWORK}" --use-private-address \
  --image "gcr.io/google.com/cloudsdktool/cloud-sdk:{CLOUD_SDK_TAG}" \
  --name "grm-rare-filter" \
  --machine-type "{FILTER_MACHINE}" --disk-size "{FILTER_DISK_GB}" \
  --input  PLINK2="{PLINK2_BIN_GS}" \
  --input  PLINK="{PLINK_BIN_GS}" \
  --output-recursive RARE_DIR="{RARE_OUT_GS}" \
  --env PANEL_GS="{PANEL_GS}" \
  --env KEEP_GS="{KEEP_GS}" \
  --env BED_RARE_NAME="{BED_RARE_NAME}" \
  --command '
    set -eo pipefail
    chmod +x "$PLINK2" "$PLINK"

    # Download unified panel (pgen/pvar/psam) and keep list
    mkdir -p /tmp/panel
    gcloud storage cp "$PANEL_GS.pgen" /tmp/panel/gwpanel.pgen
    gcloud storage cp "$PANEL_GS.pvar" /tmp/panel/gwpanel.pvar
    gcloud storage cp "$PANEL_GS.psam" /tmp/panel/gwpanel.psam
    gcloud storage cp "$KEEP_GS"       /tmp/panel/keep.txt
    echo "download complete"
    echo "pgen: $(du -sh /tmp/panel/gwpanel.pgen | cut -f1)"

    # QC and filter: rare variants (MAF 0.01 %–1 %), biallelic, tight call rate
    "$PLINK2" \
      --pfile /tmp/panel/gwpanel \
      --keep  /tmp/panel/keep.txt \
      --nonfounders \
      --maf 0.0001 --max-maf 0.01 \
      --hwe 1e-10 keep-fewhet \
      --geno 0.01 \
      --max-alleles 2 \
      --threads $(nproc) \
      --make-bed \
      --out "$RARE_DIR/$BED_RARE_NAME"

    echo "variants after rare filter: $(wc -l < \"$RARE_DIR/$BED_RARE_NAME.bim\")"
    echo "samples: $(wc -l < \"$RARE_DIR/$BED_RARE_NAME.fam\")"

    # Allele frequencies for GRM shards (plink 1.9 .frq format)
    "$PLINK" \
      --bfile "$RARE_DIR/$BED_RARE_NAME" \
      --freq \
      --out "$RARE_DIR/${BED_RARE_NAME}_freq"

    echo "freq file written"
    echo "BED size: $(du -sh \"$RARE_DIR/$BED_RARE_NAME.bed\" | cut -f1)"
  ' 2>&1 | tee /tmp/rare_filter.log
tail -5 /tmp/rare_filter.log
"""], check=True)

## Monitor filter job

In [ ]:
subprocess.run(["bash", "-c", f"""
dstat --provider google-batch \
  --project "{PROJECT_ID}" --location "{REGION}" \
  --jobs "grm-rare-filter*" --status '*' --full
"""], check=False)

## Preflight check

Run after step 1 completes. **Update `BED_SIZE_RARE_GB` in config with the actual BED size**
before submitting GRM shard jobs — memory allocation depends on it.

In [ ]:
subprocess.run(["bash", "-c", f"""
echo "=== rare panel files ==="
for ext in bed bim fam; do
  gcloud storage ls -l "{RARE_OUT_GS}/{BED_RARE_NAME}.$ext" 2>/dev/null \
    || echo "  MISSING: {BED_RARE_NAME}.$ext"
done

echo
echo "=== variant count ==="
gcloud storage cat "{RARE_OUT_GS}/{BED_RARE_NAME}.bim" 2>/dev/null | wc -l \
  || echo "  (could not read .bim)"

echo
echo "=== sample count ==="
gcloud storage cat "{RARE_OUT_GS}/{BED_RARE_NAME}.fam" 2>/dev/null | wc -l \
  || echo "  (could not read .fam)"

echo
echo "=== BED size — update BED_SIZE_RARE_GB in config to this value ==="
gcloud storage ls -l "{RARE_OUT_GS}/{BED_RARE_NAME}.bed" 2>/dev/null \
  | awk '{{printf "  %.1f GB\n", $1/1024/1024/1024}}' || true

echo
echo "=== frequencies ==="
gcloud storage ls -l "{RARE_OUT_GS}/{BED_RARE_NAME}_freq.frq" 2>/dev/null \
  || echo "  MISSING"
"""], check=True)

## Step 2: Build GRM shard tasks

In [ ]:
shards_per_task = N_SHARDS // N_TASKS
tasks_tsv = "/tmp/grm_rare_tasks.tsv"
with open(tasks_tsv, "w") as fh:
    fh.write("--env TASK_SHARDS\n")
    for t in range(N_TASKS):
        shards = list(range(t * shards_per_task + 1,
                             (t + 1) * shards_per_task + 1))
        fh.write(",".join(map(str, shards)) + "\n")
print(f"tasks written: {tasks_tsv}")
print(open(tasks_tsv).read())

## Submit GRM shard jobs

In [ ]:
subprocess.run(["bash", "-c", f"""
set -eo pipefail
DSUB_DIR=$(python -c "import dsub, os; print(os.path.dirname(dsub.__file__))")
sed -i -E \
  "s|cloud-sdk:[0-9]+\\.[0-9]+\\.[0-9]+-slim|cloud-sdk:{CLOUD_SDK_TAG}|g" \
  "$DSUB_DIR/providers/google_utils.py"

dsub \
  --provider google-batch --project "{PROJECT_ID}" --regions "{REGION}" \
  --logging "{LOG_GS}" \
  --service-account "{SERVICE_ACCOUNT}" \
  --network "{NETWORK}" --subnetwork "{SUBNETWORK}" --use-private-address \
  --image "gcr.io/google.com/cloudsdktool/cloud-sdk:{CLOUD_SDK_TAG}" \
  --name "grm-rare-shards" \
  --machine-type "{SHARD_MACHINE}" --disk-size "{SHARD_DISK_GB}" \
  --input        PLINK_BIN="{PLINK_BIN_GS}" \
  --input-recursive  BED_DIR="{RARE_OUT_GS}" \
  --output-recursive SHARD_DIR="{SHARD_RARE_GS}" \
  --env BED_NAME="{BED_RARE_NAME}" \
  --env N_SHARDS="{N_SHARDS}" \
  --env PLINK_MEM_MB="{PLINK_MEM_MB}" \
  --tasks /tmp/grm_rare_tasks.tsv \
  --command '
    set -eo pipefail
    chmod +x "$PLINK_BIN"
    BED_PREFIX="$BED_DIR/$BED_NAME"
    FREQ_PATH="$BED_DIR/${{BED_NAME}}_freq.frq"
    IFS="," read -ra SHARDS <<< "$TASK_SHARDS"
    for k in "${{SHARDS[@]}}"; do
      echo "--- shard $k / $N_SHARDS ---"
      "$PLINK_BIN" \
        --bfile "$BED_PREFIX" \
        --read-freq "$FREQ_PATH" \
        --make-grm-bin --parallel "$k" "$N_SHARDS" \
        --memory "$PLINK_MEM_MB" \
        --out "$SHARD_DIR/grm.shard$k"
      rm -f "$SHARD_DIR/grm.shard$k.grm.N.bin"
    done
  ' 2>&1 | tee /tmp/grm_rare_jobs.log
cat /tmp/grm_rare_jobs.log
"""], check=True)

## Monitor shard jobs

In [ ]:
subprocess.run(["bash", "-c", f"""
dstat --provider google-batch \
  --project "{PROJECT_ID}" --location "{REGION}" \
  --jobs "grm-rare-shards*" --status '*' --full
"""], check=False)

## Copy notebook to bucket

In [ ]:
import subprocess, os
_nb = os.path.expanduser('~/repos/AOU-covariance/notebooks/extra/grm_rare_shards.ipynb')
_gs = f'{SHARD_RARE_GS}/notebooks/grm_rare_shards.ipynb'
subprocess.run(['gcloud', 'storage', 'cp', _nb, _gs], check=True)
print(f'notebook -> {_gs}')